# 🚀 머신러닝 실습 : 고객 구매 데이터로 성별 예측 모델링 (분류 문제)

* 주어진 데이터는 백화점 고객의 1년 간 구매 데이터입니다.
* 고객 3,500명에 대한 학습용 데이터(y.csv, X.csv)를 이용하여 성별예측 모형을 만들어보세요.
* 모델의 성능은 자유롭게 측정해봅니다!

## [실습 프로세스]
1. 데이터 불러오기  
2. 데이터 탐색
3. 데이터 전처리  
4. 학습/테스트 데이터 분리  
5. 모델 선택 및 학습  
6. 예측 및 평가  

# 0. 라이브러리 불러오기
* 라이브러리를 가져와서 과정을 준비합니다

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

import os # 파일 시스템 경로 작업을 위해 (CSV 파일 로드 시 유용)

# 데이터 분할 및 모델 선택 도구
from sklearn.model_selection import train_test_split, GridSearchCV

# 전처리 도구
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# 머신러닝 모델
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# 모델 평가 지표
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


# 1. 데이터 불러오기
* 데이터를 가져와서 과정을 준비합시다.
- 인코딩 방식은 'euc-kr' 을 활용하세요.
- 데이터 출처 : 한국데이터산업진흥원 빅데이터분석기사 실기 공개 예시 문항

- 독립 변수 데이터셋 : ./data/X.csv
- 종속 변수 데이터셋 : ./data/y.csv


데이터 파일을 불러옵니다. 보통 CSV 파일을 pandas로 읽어옵니다.

In [2]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: c:\githome\hipython_rep


In [3]:
X = pd.read_csv('./data1/X.csv', encoding='euc-kr')
y = pd.read_csv('./data1/y.csv', encoding='euc-kr')


# 2. 데이터 탐색하기
* 데이터를 이해할 수 있도록 탐색과정을 수행해봅시다.

- 데이터의 상위 몇 개 행을 출력하여 전체 구조를 미리 확인합니다.
- 데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.
- 데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

In [4]:
X.info(), y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cust_id  3500 non-null   int64  
 1   총구매액     3500 non-null   int64  
 2   최대구매액    3500 non-null   int64  
 3   환불금액     1205 non-null   float64
 4   주구매상품    3500 non-null   object 
 5   주구매지점    3500 non-null   object 
 6   내점일수     3500 non-null   int64  
 7   내점당구매건수  3500 non-null   float64
 8   주말방문비율   3500 non-null   float64
 9   구매주기     3500 non-null   int64  
dtypes: float64(3), int64(5), object(2)
memory usage: 273.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   cust_id  3500 non-null   int64
 1   gender   3500 non-null   int64
dtypes: int64(2)
memory usage: 54.8 KB


(None, None)

In [5]:
X.head(3), y.head(3)

(   cust_id      총구매액     최대구매액       환불금액   주구매상품 주구매지점  내점일수   내점당구매건수  \
 0        0  68282840  11264000  6860000.0      기타   강남점    19  3.894737   
 1        1   2136000   2136000   300000.0     스포츠   잠실점     2  1.500000   
 2        2   3197000   1639000        NaN  남성 캐주얼   관악점     2  2.000000   
 
      주말방문비율  구매주기  
 0  0.527027    17  
 1  0.000000     1  
 2  0.000000     1  ,
    cust_id  gender
 0        0       0
 1        1       0
 2        2       1)

In [ ]:
X = X.drop('cust_id', axis = 1)


KeyError: "['cust_id'] not found in axis"

In [8]:
y = y.drop('cust_id', axis = 1)


# 3. 데이터 전처리
* 전처리 과정을 통해서 머신러닝에 사용할 수 있는 형태의 데이터 준비

필요한 라이브러리를 불러옵니다.
- 인코딩 : LabelEncoder
- 데이터 표준화 : StandardScaler
* 단순히 1부터의 숫자를 부여한 'cust_id'를 수치형 변수로 받아들이면, 결과가 왜곡될 수 있으니 컬럼을 제거합니다.
- 데이터에 결측치가 있는지 확인해보세요
- 결측치에 0으로 채워 넣어 모델 학습에 지장이 없도록 합니다.
- 문자형 범주 데이터를 숫자로 바꾸기 위한 인코딩을 수행합니다.

각 데이터에 표준화를 적용하여 데이터의 스케일(크기 차이)을 맞춰줍니다.
- 평균을 0, 표준편차를 1로 맞춰서 → 데이터가 정규 분포 형태로 변환되도록 하세요

In [10]:
pd.isnull(X).sum()

총구매액          0
최대구매액         0
환불금액       2295
주구매상품         0
주구매지점         0
내점일수          0
내점당구매건수       0
주말방문비율        0
구매주기          0
dtype: int64

In [12]:
X = X.fillna(0)

In [14]:
X_encoded = pd.get_dummies(X, columns=['주구매상품', '주구매지점'], dtype=int)

In [15]:
x_train, x_test, y_train, y_test = train_test_split(X_encoded, 
                 y, 
                 test_size= 0.2, 
                 random_state= 11)

In [16]:
scaler1 = StandardScaler()
scaler1.fit(x_train)
x_train_scaled = scaler1.transform(x_train)
x_test_scaled = scaler1.transform(x_test)


# 5-1. 모델링 - LogisticRegression

* 본격적으로 모델을 선언하고 학습시킵니다.
- 필요한 라이브러리를 불러옵니다.
- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.

In [17]:
model1 = LogisticRegression(random_state= 11)
model1.fit(x_train_scaled, y_train)
pred1 = model1.predict(x_test_scaled)
accuracy_score(y_test, pred1)


0.6414285714285715



# 6-1. 예측 성능 확인해보기 - LogisticRegression

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [18]:
confusion_matrix(y_test, pred1)
print(classification_report(y_test, pred1))

              precision    recall  f1-score   support

           0       0.66      0.87      0.75       439
           1       0.54      0.26      0.35       261

    accuracy                           0.64       700
   macro avg       0.60      0.56      0.55       700
weighted avg       0.62      0.64      0.60       700




# 5-2. 모델링 - DecisionTreeClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.


필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [19]:
model2 = DecisionTreeClassifier()
model2.fit(x_train_scaled, y_train)
pred2 = model2.predict(x_test_scaled)
accuracy_score(y_test, pred2)

0.5871428571428572



<br/>
<br/>

# 6-2. 예측 성능 확인해보기 - DecisionTreeClassifier

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [20]:
confusion_matrix(y_test, pred2)
print(classification_report(y_test, pred2))

              precision    recall  f1-score   support

           0       0.68      0.66      0.67       439
           1       0.45      0.47      0.46       261

    accuracy                           0.59       700
   macro avg       0.56      0.56      0.56       700
weighted avg       0.59      0.59      0.59       700




# 5-3. 모델링 - RandomForestClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.



필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [28]:
model3 = RandomForestClassifier(random_state=0, max_depth=8)
param = { 
    'max_depth': [11,12,13], 'min_samples_split' : [19,20,21]

}
grid_dtree = GridSearchCV(model3, param_grid=param, cv = 3, refit=True)
grid_dtree.fit(x_train_scaled, y_train)
pred3 = grid_dtree.predict(x_test_scaled)
accuracy_score(y_test, pred3)

0.6614285714285715


# 6-3. 예측 성능 확인해보기 - RandomForestClassifier

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [22]:
confusion_matrix(y_test, pred3)
print(classification_report(y_test, pred3))

              precision    recall  f1-score   support

           0       0.67      0.89      0.76       439
           1       0.58      0.25      0.35       261

    accuracy                           0.65       700
   macro avg       0.62      0.57      0.55       700
weighted avg       0.63      0.65      0.61       700




# 5-4. 모델링 - XGBoost

* 본격적으로 모델을 선언하고 학습시킵니다.



필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [23]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)  # Series → 1D array
y_test_encoded = le.transform(y_test)

In [27]:
xgb_model = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3, use_label_encoder=False)
xgb_model.fit(x_train_scaled, y_train_encoded)
xgb_predictions = xgb_model.predict(x_test_scaled)
accuracy_score(y_test_encoded,xgb_predictions)

0.6414285714285715



<br/>
<br/>

# 6-4. 예측 성능 확인해보기 - XGBoost

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [26]:
confusion_matrix(y_test_encoded, xgb_predictions)
print(classification_report(y_test_encoded, xgb_predictions))

              precision    recall  f1-score   support

           0       0.67      0.75      0.71       439
           1       0.47      0.37      0.42       261

    accuracy                           0.61       700
   macro avg       0.57      0.56      0.56       700
weighted avg       0.60      0.61      0.60       700





# 7.  위 4가지 모델의 학습 & 예측 & 평가 결과를 확인하고 최고 성능을 내는 모델을 찾아봅시다!

- 어떤 모델이 가장 성능이 좋은가요 ?